# This is the first code in this repo where we will explore Langgraph's capabilities

In [1]:
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
api_key= os.getenv("GROQ_API_KEY")

llm = ChatGroq(model = "llama-3.1-8b-instant", api_key = api_key)

In [3]:
class TravelPlan(BaseModel):
    destination: str = Field(..., description="City or country name")
    date: str = Field(..., description="Travel date in YYYY-MM-DD format")
    budget: str = Field(..., description="Approximate budget")
    weather: str = Field(..., description="Weather forecast")
    hotel: str = Field(..., description="Suggested hotel with price")

parser = PydanticOutputParser(pydantic_object=TravelPlan)

In [4]:
def get_weather(destination: str, date: str) -> str:
    """Mock weather API"""
    return f"Sunny, 28°C in {destination} on {date}"

def suggest_hotel(destination: str, budget: str) -> str:
    """Mock hotel recommender"""
    if "low" in budget.lower():
        return f"Budget Inn in {destination}, $50/night"
    elif "high" in budget.lower():
        return f"Luxury Suites in {destination}, $250/night"
    return f"Comfort Stay in {destination}, $120/night"

In [5]:
class State(dict):
    text: str
    destination: str
    date: str
    budget: str
    weather: str
    hotel: str

In [6]:
def extract_request(state: State):
    """Use LLM to extract structured request"""
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.environ["GROQ_API_KEY"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a travel planner AI. Extract fields into structured output."),
        ("human", "{text}"),
        ("system", "Format your response as:\n{format_instructions}")
    ])
    chain = prompt.partial(format_instructions=parser.get_format_instructions()) | llm | parser
    result = chain.invoke({"text": state["text"]})
    return {
        "destination": result.destination,
        "date": result.date,
        "budget": result.budget,
        "weather": "",  # placeholder
        "hotel": ""     # placeholder
    }

In [7]:
def call_weather_tool(state: State):
    return {"weather": get_weather(state["destination"], state["date"])}

In [8]:
def call_hotel_tool(state: State):
    return {"hotel": suggest_hotel(state["destination"], state["budget"])}

In [12]:
def finalize_output(state: State):
    plan = TravelPlan(
        destination=state["destination"],
        date=state["date"],
        budget=state["budget"],
        weather=state["weather"],
        hotel=state["hotel"]
    )
    print("\nFinal Structured Travel Plan:")
    print(plan)
    return plan

In [13]:
graph = StateGraph(State)

graph.add_node("extract_request", extract_request)
graph.add_node("call_weather_tool", call_weather_tool)
graph.add_node("call_hotel_tool", call_hotel_tool)
graph.add_node("finalize_output", finalize_output)

graph.set_entry_point("extract_request")
graph.add_edge("extract_request", "call_weather_tool")
graph.add_edge("call_weather_tool", "call_hotel_tool")
graph.add_edge("call_hotel_tool", "finalize_output")
graph.add_edge("finalize_output", END)

In [14]:
app = graph.compile()

user_input = "I want to travel to Paris next Friday with a medium budget."
result = app.invoke({"text": user_input})

OutputParserException: Invalid json output: Based on your preferences, I have extracted the following information:

```
{
  "destination": "Paris",
  "date": "2023-12-15",  // Assuming next Friday is 2023-12-15
  "budget": "Medium",
  "weather": "Partly cloudy with a high of 12°C and a low of 8°C",
  "hotel": "Ibis Paris Gare de Lyon Reuilly (approx. $120 per night)"
}
```

Note: The budget has been categorized as 'Medium' based on the assumption that you are looking for a moderate expenditure. If you have a specific budget in mind, please let me know and I can update the information accordingly.

Also, the weather forecast is based on the assumption that you will be traveling on December 15, 2023. Please note that weather forecasts can change, and it's always a good idea to check the latest forecast before your trip.

As for the hotel suggestion, Ibis Paris Gare de Lyon Reuilly is a 3-star hotel located near the Gare de Lyon train station, which offers easy access to the city center. 

## I have kept this error as it is because I want you to know that this happens all the time but you should not worry and lets resolve this!

### Whats this? Output Parser Error
### How to fix? Enhancing Prompt will work.

In [16]:
#Check out the changes prompt carefully

def new_extract_request(state: State):
    """Use LLM to extract structured request"""
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.environ["GROQ_API_KEY"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a travel planner AI. Extract the information strictly in JSON."),
        ("human", "{text}"),
        ("system", "Return only valid JSON that matches this schema:\n{format_instructions}\n\n"
        "Do not add explanations, notes, or extra text. No markdown, no comments, no natural language.")
        ])

    chain = prompt.partial(format_instructions=parser.get_format_instructions()) | llm | parser
    result = chain.invoke({"text": state["text"]})
    return {
        "destination": result.destination,
        "date": result.date,
        "budget": result.budget,
        "weather": "",  # placeholder
        "hotel": ""     # placeholder
    }

In [17]:
graph = StateGraph(State)

graph.add_node("extract_request", new_extract_request)
graph.add_node("call_weather_tool", call_weather_tool)
graph.add_node("call_hotel_tool", call_hotel_tool)
graph.add_node("finalize_output", finalize_output)

graph.set_entry_point("extract_request")
graph.add_edge("extract_request", "call_weather_tool")
graph.add_edge("call_weather_tool", "call_hotel_tool")
graph.add_edge("call_hotel_tool", "finalize_output")
graph.add_edge("finalize_output", END)

In [18]:
app = graph.compile()

user_input = "I want to travel to Paris next Friday with a medium budget."
result = app.invoke({"text": user_input})


Final Structured Travel Plan:
destination='Paris' date='2024-09-13' budget='medium' weather='Sunny, 28°C in Paris on 2024-09-13' hotel='Comfort Stay in Paris, $120/night'


## It worked!